# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a practical template for loading and exploring the FAIR<sup>2</sup> dataset using the `mlcroissant` library. We will use the Croissant metadata schema and explore the structure and content of the dataset using the official identifiers (`@id`) for all entities, ensuring clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}\n\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

`mlcroissant` exposes the dataset structure via the `.record_sets` attribute. Each record set and field within should be referenced by their `@id`.

In [ ]:
# List all available record sets and their fields/columns by @id
print("Available record sets (by @id):\n")
for record_set in dataset.record_sets:
    print(f"RecordSet @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    if hasattr(record_set, 'description') and record_set.description:
        print(f"  Description: {record_set.description}")
    print("  Fields / Columns by @id:")
    for field in record_set.fields:
        print(f"    Field @id: {field.id} -- name: {field.name} -- type: {getattr(field, 'data_type', None)}")
    print('-'*60)

# Example: preview first 2 records of each RecordSet
for record_set in dataset.record_sets:
    print(f"RecordSet @id: {record_set.id}")
    for i, rec in enumerate(dataset.records(record_set=record_set.id)):
        print(rec)
        if i == 1:
            break
    print()

## 3. Data Extraction
Let's load records from each record set (referenced by their `@id`) into Pandas DataFrames. This makes it easy to work with the data programmatically.

> **Note:** Select the appropriate record set(s) for your analytic task. We'll demonstrate with the first available record set.

In [ ]:
# Extract all records from each record set (by @id)
# You can modify the list to select only certain record sets
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet @id: {record_set_id} -- {df.shape[0]} rows × {df.shape[1]} columns\n")
    print(f"Fields: {df.columns.tolist()}")
    print(df.head(2))
    print('-'*80)

# For demonstration, pick the first record set for further analysis below:
main_record_set_id = record_set_ids[0] if record_set_ids else None

# Preview columns of the main DataFrame
if main_record_set_id:
    print(f"Columns in main DataFrame ({main_record_set_id}): {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Now, apply common data processing and filtering steps. 

Let's demonstrate:
- Filtering by a numeric field (e.g., 'coverage(%)')
- Normalizing this field
- Grouping by a categorical field, if present

Please use the correct `@id` for the field you wish to analyze (see previous cell). Adjust field names as needed.

In [ ]:
# Example field names -- replace with exact @id fields as per the chosen record set
import numpy as np
pd.set_option('mode.use_inf_as_na', True)

record_set_id = main_record_set_id
if record_set_id is not None:
    df = dataframes[record_set_id].copy()

    # Try common protein analysis columns: coverage(%), MW(kDa), peptide counts, etc.
    numeric_field_id = None
    possible_numeric_fields = [col for col in df.columns if col.lower().startswith('coverage') or col.lower().startswith('mw') or 'abundance' in col.lower() or 'count' in col.lower() or 'quant' in col.lower()]
    if possible_numeric_fields:
        numeric_field_id = possible_numeric_fields[0]
    else:
        numeric_field_id = df.columns[0]  # fallback

    print(f"Using numeric field: {numeric_field_id} (for demonstration)")

    # Remove missing/non-numeric values if any
    df_num = df.copy()
    df_num[numeric_field_id] = pd.to_numeric(df_num[numeric_field_id], errors='coerce')
    df_num = df_num.dropna(subset=[numeric_field_id])

    threshold = df_num[numeric_field_id].median() if not df_num.empty else 0

    filtered_df = df_num[df_num[numeric_field_id] > threshold]
    print(f"Records where {numeric_field_id} > {threshold:.3f} (count={len(filtered_df)}):")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} (Z-score):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field: e.g., 'description', 'gene', etc.
    group_field_candidates = [col for col in df.columns if col.lower() in ['description', 'gene', 'sample', 'accession']]
    group_field_id = group_field_candidates[0] if group_field_candidates else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"Grouped mean {numeric_field_id} by {group_field_id} (top 5):")
        display(grouped_df.head())

## 5. Visualization
Let's plot the distribution of the selected numeric field.

> Adjust field names as appropriate for the actual dataset columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id and not filtered_df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True, color='steelblue')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    # If grouping field is available, show boxplot
    if group_field_id:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
- We loaded the FAIR<sup>2</sup> dataset from its Croissant schema using `mlcroissant`.
- The notebook demonstrates how to systematically reference entities by their `@id` and explore and process tabular data.
- We applied basic EDA operations, such as filtering, normalization, and grouping, and visualized the distribution of key numeric fields.

Continue the analysis by mapping the exact scientific variables (e.g., protein abundance, modification scores) and their `@id`s to harness the dataset for domain-specific questions.